In [ ]:
import os
import gc
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

pd.set_option('display.max_columns', 500)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [ ]:
# путь до данных на компьютере
path = 'traindata/'

In [ ]:
# прописать свой путь
path = ''

In [ ]:
def reduce_mem_usage(df, int_cast=True, obj_to_category=True, subset=None):
    """
    Iterate through all the columns of a dataframe and modify the data type to reduce memory usage.
    :param df: dataframe to reduce (pd.DataFrame)
    :param int_cast: indicate if columns should be tried to be casted to int (bool)
    :param obj_to_category: convert non-datetime related objects to category dtype (bool)
    :param subset: subset of columns to analyse (list)
    :return: dataset with the column dtypes adjusted (pd.DataFrame)
    """
    start_mem = df.memory_usage().sum() / 1024 ** 2;
    gc.collect()
    print('Memory usage of dataframe is {:.2f} MB'.format(start_mem))

    cols = subset if subset is not None else df.columns.tolist()

    for col in tqdm(cols):

        col_type = df[col].dtype

        if col_type != object and col_type.name != 'category' and 'datetime' not in col_type.name:
            c_min = df[col].min()
            c_max = df[col].max()

            # test if column can be converted to an integer
            treat_as_int = str(col_type)[:3] == 'int'
            # if int_cast and not treat_as_int:
            #     treat_as_int = check_if_integer(df[col])

            if treat_as_int:
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.uint8).min and c_max < np.iinfo(np.uint8).max:
                    df[col] = df[col].astype(np.uint8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.uint16).min and c_max < np.iinfo(np.uint16).max:
                    df[col] = df[col].astype(np.uint16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.uint32).min and c_max < np.iinfo(np.uint32).max:
                    df[col] = df[col].astype(np.uint32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)
                elif c_min > np.iinfo(np.uint64).min and c_max < np.iinfo(np.uint64).max:
                    df[col] = df[col].astype(np.uint64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
        elif 'datetime' not in col_type.name and obj_to_category:
            df[col] = df[col].astype('category')
    gc.collect()
    end_mem = df.memory_usage().sum() / 1024 ** 2
    print('Memory usage after optimization is: {:.3f} MB'.format(end_mem))
    print('Decreased by {:.1f}%'.format(100 * (start_mem - end_mem) / start_mem))

    return df

In [ ]:
def read_parquet_dataset_from_local(path_to_dataset: str, file_name_tag: str, start_from: int = 0,
                                     num_parts_to_read: int = 2, columns=None, verbose=False) -> pd.DataFrame:
    """
    читает num_parts_to_read партиций, преобразует их к pd.DataFrame и возвращает
    :param path_to_dataset: путь до директории с партициями
    :param start_from: номер партиции, с которой начать чтение
    :param num_parts_to_read: количество партиций, которые требуется прочитать
    :param columns: список колонок, которые нужно прочитать из партиции
    :return: pd.DataFrame
    """

    res = []
    dataset_paths = sorted([os.path.join(path_to_dataset, filename) for filename in os.listdir(path_to_dataset)
                              if filename.startswith(file_name_tag)])

    start_from = max(0, start_from)
    chunks = dataset_paths[start_from: start_from + num_parts_to_read]
    if verbose:
        print('Reading chunks:\n')
        for chunk in chunks:
            print(chunk)
    for chunk_path in tqdm(chunks, desc="Reading dataset with pandas"):
        print('chunk_path', chunk_path)
        chunk = pd.read_parquet(chunk_path,columns=columns)
        chunk = reduce_mem_usage(chunk)
        res.append(chunk)
        del chunk
        gc.collect()

    return pd.concat(res).fillna(0).reset_index(drop=True)

In [ ]:
def prepare_transactions_dataset(path_to_dataset: str, file_name_tag: str, num_parts_to_preprocess_at_once: int = 1, num_parts_total: int=50,
                                 save_to_path=None, verbose: bool=False):
    """
    возвращает готовый pd.DataFrame с признаками, на которых можно учить модель для целевой задачи.
    path_to_dataset: str
        путь до датасета с партициями
    num_parts_to_preprocess_at_once: int
        количество партиций, которые будут одновременно держаться в памяти и обрабатываться
    num_parts_total: int
        общее количество партиций, которые нужно обработать
    save_to_path: str
        путь до папки, в которой будет сохранен каждый обработанный блок в .parquet формате. Если None, то не будет сохранен
    verbose: bool
        логирует каждый обрабатываемый кусок данных
    """
    preprocessed_frames = []

    for step in tqdm(range(0, num_parts_total, num_parts_to_preprocess_at_once),
                                   desc="Transforming transactions data"):
        transactions_frame = read_parquet_dataset_from_local(path_to_dataset, file_name_tag, step, num_parts_to_preprocess_at_once,
                                                             verbose=verbose)
        display(transactions_frame)
        print(transactions_frame.info(max_cols=5, memory_usage='deep'))


   #здесь должен быть препроцессинг данных
   #проверялись средние/максимумы и разные способы процессинга кат фичей, в том числе катбуст энкодер
   #в соотношении между памятью и качеством выигрывает сумма по всем фичам

        prepared_df = transactions_frame[['id', 'rn']].copy()

        features = set(transactions_frame.columns) - set(['id', 'rn'])
        dummies = pd.get_dummies(transactions_frame[features], columns = features)
        prepared_df = pd.concat([prepared_df, dummies], axis=1)

        prepared_df = pd.concat([prepared_df, transactions_frame[features]], axis=1)
        agg_d = {f: 'sum' for f in set(prepared_df) - set(['id', 'rn'])}
        agg_d['rn'] = 'count'
        prepared_df = prepared_df.groupby('id').agg(agg_d).astype(int).reset_index(drop=False)

        # dummies = pd.get_dummies(prepared_df[features], columns = features)
        # prepared_df = pd.concat([prepared_df[['id', 'rn']], dummies], axis=1)
        display(prepared_df)
        prepared_df.info(max_cols=5, memory_usage='deep')


   #записываем подготовленные данные в файл
        if save_to_path:
            block_as_str = str(step)
            if len(block_as_str) == 1:
                block_as_str = '00' + block_as_str
            else:
                block_as_str = '0' + block_as_str
            prepared_df.to_parquet(os.path.join(save_to_path, f'processed_chunk_{block_as_str}.parquet'))
        gc.collect()
        # preprocessed_frames.append(transactions_frame)


    # return pd.concat(preprocessed_frames)

In [ ]:
data = prepare_transactions_dataset(path, 'train', num_parts_to_preprocess_at_once=1, num_parts_total=12,
                                    save_to_path='traindata/')

In [ ]:
data = read_parquet_dataset_from_local('traindata/', 'processed', 0, 12, verbose=1)

In [ ]:
# пример полученных данных
data.head()

In [ ]:
# объем данных
data.info(max_cols=5, memory_usage='deep')

In [ ]:
# добавим значения целевой переменной
targets = pd.read_csv(path + 'train_target.csv')
targets

In [ ]:
train_data_target = data.merge(targets, on="id")

In [ ]:
train_data_target.shape, data.shape

In [ ]:
del data
gc.collect()

In [ ]:
# убедимся что нет нанов
train_data_target.isna().sum().sum()

In [ ]:
train_data_target.columns

In [ ]:
# подготовим фичи и таргет
train_data = train_data_target.drop(['flag', 'id'], axis=1)
train_labels = train_data_target['flag']
del train_data_target

In [ ]:
# посмотрим что нет колонок с 1 значением
train_data.nunique()

In [ ]:
# катбуст лучше всего оптимизирован - с ним легче вывозить по памяти
!pip install catboost

In [ ]:
from catboost import cv
from catboost import CatBoostClassifier

In [ ]:
# подбор через перебор параметров, оставляю код только для l2_leaf_reg
tree_params = {
    "objective": "Logloss",
    "eval_metric": "AUC",
    "l2_leaf_reg": 0.5,
    "learning_rate": 0.6,
    "n_estimators": 1000,
    "early_stopping_rounds": 50
}

In [ ]:
# с увеличением количества эстиматоров и добавлении новых фичей - например максимальное по атрибутам -> можно улучшить качество
# но при таком обучении на ресурсах колаба будет обучаться около ~3-5 часов
model = CatBoostClassifier(**tree_params)

In [ ]:
# здесь можно добавлять/перебирать разные наборы параметров и их значений
grid = {'l2_leaf_reg': [0.1, 0.01]}

In [ ]:
# пример вызова grid'а
# после каждой итерации видим лучшую метрику на тесте и на обучении
# можно ориентироваться уже на метрики, полученные на тестах
# roc_auc 74 c early_stopping_ronds=50 на кросс валидации
grid_search_result = model.grid_search(grid,
                                       X=train_data,
                                       y=train_labels,
                                       cv=3,
                                       train_size=0.8)